# 第03课 - 代理设计模式

在本课中，我们将探索构建高效 AI 代理的三个基础设计模式：

1. <strong>清晰的代理指令</strong> — 制作精确的、定义角色的提示，以指导代理行为
2. **使用 Pydantic 模型的结构化输出** — 确保代理返回可预测、已验证的数据
3. <strong>单一职责代理</strong> — 设计专注的代理，每个代理专注做好一件事

我们将把每种模式应用于一个<strong>旅游目的地推荐系统</strong>场景，逐步构建一个能够推荐目的地、检查可用性和处理物流的系统。


## 设置


In [2]:
%pip install agent-framework pydantic python-dotenv --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("LLM_BASE_URL")
deployment_name = os.getenv("LLM_MODEL")

missing = [k for k, v in {
    "LLM_BASE_URL": endpoint,
    "LLM_MODEL": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"缺少必需的环境变量：{", ".join(missing)}。"
        "请将其设置为环境变量（例如在 .env 文件或 shell 环境中）。"
    )

provider = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)


## 模式1：明确的代理指令

最有影响力的模式也是最简单的：为你的代理编写清晰、详细的指令。

良好的指令应定义：
- <strong>代理是谁</strong>（角色和语气）
- <strong>代理该做什么</strong>（逐步职责）
- <strong>代理应如何表现</strong>（约束和风格）

下面，我们创建一个旅行礼宾代理，带有明确的指令来塑造它生成的每个回复。


In [2]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""你是一位名叫 Alex 的豪华旅行礼宾。你的职责是：
1. 了解旅行者的偏好（预算、气候、活动）
2. 在推荐前检查目的地可用性
3. 提供详细、个性化的旅行建议
4. 始终提及签证要求和最佳旅行季节
请保持热情、专业，并对旅行充满热忱。""",
)

response = await agent.run(
    "我想去一个美食和历史都很棒的地方度一周假。预算大约 2500 美元。"
)
print(f"智能体：{response}")


智能体：你好！我是 Alex，非常荣幸能为你策划这趟旅程！一周的美食与历史 immersion（沉浸之旅）——这简直是灵魂与味蕾的双重盛宴，我已经开始兴奋了！🍷✨

为了给你打造一份精准且不留遗憾的行程，我需要先确认几个关键细节（这些会直接决定我们的目的地筛选）：

1. **你的出发城市是哪里？** 这会影响航班成本与飞行时间。
2. **这 2500 美元预算是否包含国际往返机票？** （如果是全包预算，我们需要锁定高性价比的顶级目的地；如果不含机票，选择面会更广，甚至可以触及欧洲的精品路线。）
3. **你预计的出行月份或日期？** 我需要确保避开雨季或极端气候，并检查当地节庆与酒店可用性。
4. **气候偏好？** 偏爱阳光温暖的氛围，还是凉爽宜人的漫步天气？

---

不过，基于你目前的需求，我已经在脑海中为你勾勒了三个**初步概念方向**——它们都以世界级美食和深厚历史著称，且在 2500 美元框架内有很强的可操作性：

### 🌮 方向一：墨西哥城（墨西哥）
*一场阿兹特克文明与当代高端美食的碰撞。*
- **美食亮点**：从街头玉米棒到 Pujol、Quintonil 等全球 50 佳餐厅，再到 Roma 区的精品咖啡馆与 Mezcal 酒吧。
- **历史亮点**：特奥蒂瓦坎金字塔、索卡洛广场、弗里达·卡罗博物馆、殖民时期建筑。
- **签证**：持美国、加拿大、日本、英国等国有效签证/护照可免签入境；中国公民需办理墨西哥旅游签证（但持美签可免签）。
- **最佳季节**：**11 月至次年 4 月**（干季，气候温和少雨，非常适合步行探索）。

### 🍷 方向二：里斯本 + 辛特拉（葡萄牙）
*大航海时代的浪漫与地中海飨宴。*
- **美食亮点**：蛋挞始祖 Pastéis de Belém、新鲜海产、杜罗河谷葡萄酒、Bacalhau（鳕鱼）的 365 种吃法。
- **历史亮点**：贝伦塔、热罗尼莫斯修道院、辛特拉皇宫、摩尔人城堡。
- **签证**：申根签证（需提前申请）。
- **最佳季节**：**4 月至 6 月** 或 **9 月至 10 月**（避开盛夏酷暑与旅游高峰，气候舒适，海鲜季正旺）。

### 🍜 方向三：清迈 + 曼谷（泰国）
*千年兰纳王国与街头米其林的美食朝圣。*
- **美食亮点**：清迈夜市、曼谷 Jay Fa

## 模式 2：使用 Pydantic 模型的结构化输出

自由形式文本对对话很有用，但下游系统需要结构化数据。
通过将 **Pydantic 模型** 与 <strong>工具函数</strong> 配对，我们可以：

- 定义代理输出的精确定义模式
- 自动验证响应
- 可靠地将代理结果集成到应用逻辑中

执行的关键是在运行代理时传递 `response_format`。这会强制
模型返回一个经过验证的 `TravelRecommendations` 对象（可通过 `response.value` 访问）
，而不是自由格式文本。`get_destination_details` 工具也返回类型化的
`DestinationRecommendation`，因此数据从始至终保持结构化。


In [4]:
class DestinationRecommendation(BaseModel):
    destination: str = ""
    available: bool = False
    best_season: str = "未知"
    highlights: list[str] = []
    estimated_budget_usd: int = 0


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation] = []
    personalized_note: str = ""


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "要查询的目的地"]
) -> DestinationRecommendation:
    """获取度假目的地的结构化详情。"""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="巴塞罗那",
            available=True,
            best_season="5-6月",
            highlights=["海滩", "建筑", "夜生活"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="东京",
            available=True,
            best_season="3-4月",
            highlights=["文化", "美食", "科技"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="开普敦",
            available=False,
            best_season="11月-3月",
            highlights=["自然", "葡萄酒", "探险"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="未知",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="你是一位旅行专家。根据旅行者的偏好推荐目的地。请使用 get_destination_details 工具。",
    tools=[get_destination_details],
)

# 传入 `response_format` 会强制智能体返回经过验证的
# TravelRecommendations 对象，而不是自由格式文本。
response = await structured_agent.run(
    "为一位热爱文化的旅行者推荐 3 个目的地，预算 2500 美元",
    options={"response_format": TravelRecommendations},
)

# Kimi 等第三方模型可能不严格遵循 response_format，
# 用 try/except 容错，避免 ValidationError 中断。
try:
    if response and response.value:
        result: TravelRecommendations = response.value
        for rec in result.recommendations:
            status = "可预订" if rec.available else "不可预订"
            print(f"{rec.destination}（{status}）")
            print(f"  最佳季节：{rec.best_season}")
            print(f"  亮点：{", ".join(rec.highlights)}")
            print(f"  预估预算：${rec.estimated_budget_usd}")
            print()
        print(f"备注：{result.personalized_note}")
    else:
        print("未返回经过验证的结构化响应。")
        print(response)
except Exception as e:
    print(f"结构化解析失败：{e}")
    print("原始响应：")
    print(response)


（不可预订）
  最佳季节：未知
  亮点：
  预估预算：$0

（不可预订）
  最佳季节：未知
  亮点：
  预估预算：$0

（不可预订）
  最佳季节：未知
  亮点：
  预估预算：$0

备注：这三座城市都提供了极高的文化密度与相对可控的旅行成本。河内的佛教宁静与法式风情、墨西哥城的古文明与现代艺术碰撞、马拉喀什的伊斯兰几何美学与撒哈拉门户地位，将分别带来亚洲、美洲与非洲三大洲截然不同的文化震撼。对于文化爱好者而言，它们都比传统的西欧大城市更具沉浸感，且预算友好得多。


## 模式 3：单一职责代理

复杂任务通过将工作拆分为多个专注的代理来执行，每个代理负责单一职责：

- 一个了解地点和可用性的 <strong>目的地专家</strong>
- 一个处理航班、酒店和行程的 <strong>物流规划师</strong>

这与软件工程中的<em>关注点分离</em>原则相呼应——每个代理都更容易独立测试、维护和改进。


In [5]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""你是一位目的地研究专家。你的唯一职责是：
1. 根据旅行者的偏好评估目的地
2. 使用提供的工具检查可用性
3. 返回一个简短的排序列表，包含优缺点
请勿讨论航班、酒店或行程安排——由另一位智能体负责。""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""你是一位旅行行程规划师。你的唯一职责是：
1. 为选定的目的地创建逐日行程
2. 在预算范围内推荐航班和酒店选项
3. 注明签证要求和旅行保险建议
请勿推荐目的地——由另一位智能体负责。""",
)

# 步骤 1：目的地专家挑选最佳选项
dest_response = await destination_agent.run(
    "我想花一周时间体验文化和美食，预算不超过 2500 美元。我该去哪里？"
)
print("=== 目的地专家 ===")
print(dest_response)

# 步骤 2：行程规划师制定旅行计划
logistics_response = await logistics_agent.run(
    f"根据以下推荐制定一周的旅行计划：\n{dest_response}"
)
print("\n=== 行程规划师 ===")
print(logistics_response)


=== 目的地专家 ===
作为目的地研究专家，为了给您最准确的建议，我需要先了解您是否有**特定偏好或感兴趣的地区**？比如亚洲、欧洲、拉丁美洲等？

同时，根据您的需求（**1周文化+美食，预算≤$2500**），以下几个目的地通常性价比高且体验丰富，您可以从中挑选1-2个，我会立即为您调取详细评估：

**亚洲**
- **泰国**（曼谷/清迈）——街头美食天堂，佛教文化深厚，物价极低
- **越南**（河内/会安/胡志明）——法越融合文化，米粉与咖啡文化出众
- **日本**（关西地区如京都/大阪）——传统文化与美食极致，预算偏紧但可行
- **马来西亚**（槟城/吉隆坡）——多元文化（华人/马来/印度），美食多样且便宜

**欧洲**
- **葡萄牙**（里斯本/波尔图）——欧洲最划算目的地之一，海鲜、蛋挞、法朵音乐
- **西班牙**（安达卢西亚地区如塞维利亚/格拉纳达）——弗拉明戈、Tapas文化，物价低于北欧

**拉丁美洲**
- **墨西哥**（墨西哥城/瓦哈卡）——玛雅/阿兹特克文化、街头玉米美食、色彩艺术
- **秘鲁**（利马/库斯科）——印加文明、世界级料理（世界最佳餐厅之一），预算可控

**其他**
- **摩洛哥**（马拉喀什/非斯）——北非异域风情、香料市集、传统庭院（Riad）
- **格鲁吉亚**（第比利斯）——高加索文化、葡萄酒发源地、烤肉与奶酪，性价比极高

**请告诉我您倾向的2-3个目的地**，我将立即为您查询结构化详情，并返回一份带优缺点的排序清单！

=== 行程规划师 ===
您好！我已收到这些精彩的目的地推荐。根据我的职责分工，**我需要您明确选定1-2个具体目的地后**，才能为您立即启动详细的行程规划。

请从上述列表中直接告诉我您的最终决定（例如："我选择泰国曼谷/清迈" 或 "我选择葡萄牙里斯本/波尔图"）。

一旦您确认目的地，我将为您呈上包含以下内容的完整方案：
- **逐日行程**：7天文化与美食路线，含具体景点、餐厅与体验
- **航班与酒店推荐**：严格控制在 **≤$2,500** 总预算内的最优组合
- **签证要求**：中国公民（或您适用的国籍）入境规定
- **旅行保险建议**：针对该地区与活动的保障方案

**请回复您选定的目的地，我马上开始规划！**


## 总结

在本课中，我们将三个主动设计模式应用于旅游推荐场景：

| 模式 | 关键思想 | 优势 |
|---|---|---|
| <strong>明确指令</strong> | 预先定义角色、职责和约束 | 保持一致、符合品牌形象的代理行为 |
| <strong>结构化输出</strong> | 使用Pydantic模型作为响应格式 | 经过验证、机器可读的结果 |
| <strong>单一职责</strong> | 让每个代理专注于一项工作 | 更易测试、维护和组合 |

这些模式自然组合——你可以将明确指令与结构化输出结合到单一职责代理中，构建健壮、适合生产的系统。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
